In [1]:
import os
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from ultralytics import YOLO

# === CONFIG ===
input_txt = r"C:\Users\rtvan\Documents\GitHub\AI-Waste_Vision_POC\Bijplaatsing Detection\Yolov8\container_images.txt"
dataset_root = r"C:/ai-semester/routeeventlog"
output_txt = r"C:\Users\rtvan\Documents\GitHub\AI-Waste_Vision_POC\Bijplaatsing Detection\Yolov8\bijplaatsing_images.txt"

# YOLO model laden
model = YOLO("runs/classify/bijplaatsing_classification_v3/weights/best.pt")
class_names = ["met_bijplaatsing", "zonder_bijplaatsing"]

print("Model geladen.")


Model geladen.


In [3]:
file_index = {}  # filename → absoluut pad
for dirpath, _, files in os.walk(dataset_root):
    for fname in files:
        if fname.lower().endswith(".jpg"):
            file_index[fname] = os.path.join(dirpath, fname)

print(f"Totaal gevonden afbeeldingen: {len(file_index)}")

# TXT inladen
with open(input_txt, "r") as f:
    raw_paths = [line.strip() for line in f.readlines()]

print(f"{len(raw_paths)} paden ingelezen uit container_images.txt")

# Classificeren en wegschrijven
with open(output_txt, "w") as out_f:
    for p in tqdm(raw_paths, desc="Classifying"):
        filename = os.path.basename(p)

        if filename not in file_index:
            print(f"⚠️ Bestand niet gevonden: {filename}")
            continue

        real_path = file_index[filename]

        result = model.predict(real_path, verbose=False)[0]
        pred_idx = result.probs.top1
        label = class_names[pred_idx]

        if label == "met_bijplaatsing":

            # 🔥 maak pad relatief i.p.v. absoluut
            rel_path = real_path.replace("C:/ai-semester/", "").replace("C:\\ai-semester\\", "")

            # Windows backslashes toevoegen (optioneel)
            rel_path = rel_path.replace("/", "\\")

            out_f.write(rel_path + "\n")

print("Klaar! Nieuwe TXT met relative paths aangemaakt.")

Totaal gevonden afbeeldingen: 90498
20581 paden ingelezen uit container_images.txt


Classifying:   0%|          | 6/20581 [00:00<06:02, 56.81it/s]

⚠️ Bestand niet gevonden: image_path


Classifying: 100%|██████████| 20581/20581 [08:51<00:00, 38.70it/s]

Klaar! Nieuwe TXT met relative paths aangemaakt.


In [ ]:
# Toon 20 willekeurige foto's uit de dataset + voorspelling
sample_files = random.sample(list(file_index.values()), 20)

fig = plt.figure(figsize=(15, 10))

for idx, img_path in enumerate(sample_files):
    result = model.predict(img_path, verbose=False)[0]
    pred_idx = result.probs.top1
    conf = float(result.probs.top1conf)
    label = class_names[pred_idx]

    img = image.load_img(img_path, target_size=(224, 224))

    ax = fig.add_subplot(4, 5, idx + 1)
    ax.imshow(img)
    ax.set_title(f"{label}\nconf: {conf:.2f}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Paden inlezen
with open(output_txt, "r") as f:
    bijplaatsing_paths = [line.strip() for line in f.readlines()]

print(f"Totaal {len(bijplaatsing_paths)} voorspeld als bijplaatsing.")

# Random subset kiezen
sample = random.sample(bijplaatsing_paths, min(20, len(bijplaatsing_paths)))

# Plotten
fig = plt.figure(figsize=(15, 10))

for idx, img_path in enumerate(sample):
    result = model.predict(img_path, verbose=False)[0]
    conf = float(result.probs.top1conf)

    img = image.load_img(img_path, target_size=(224, 224))

    ax = fig.add_subplot(4, 5, idx + 1)
    ax.imshow(img)
    ax.set_title(f"Bijplaatsing\nconf: {conf:.2f}")
    ax.axis("off")

plt.tight_layout()
plt.show()